[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/02_keras_text/02_keras_text_solutions.ipynb)

# 02. Keras 텍스트 분류 — 연습 문제 해설

[02_keras_text.ipynb](https://colab.research.google.com/github/karzit/temp/blob/master/notebooks/text-classification-practice/02_keras_text/02_keras_text.ipynb) 끝의 연습 문제 6개에 대한 정답 코드와 해설입니다.
**먼저 직접 시도해본 뒤** 참고하세요.

> **신경망은 실행할 때마다 결과가 조금씩 달라집니다.** seed를 고정해도 환경(CPU/GPU, TensorFlow 버전)에
> 따라 소수점 이하가 달라집니다. 아래 숫자는 방향을 보는 용도이지 정답이 아닙니다.

In [ ]:
import os
import sys
import time

IN_COLAB = "google.colab" in sys.modules
BASE_URL = "https://raw.githubusercontent.com/karzit/temp/master/notebooks/text-classification-practice/data"

if IN_COLAB:
    !pip install -q pandas scikit-learn matplotlib koreanize-matplotlib
    for _f in ["02_train.csv", "02_test_x.csv", "02_test_y.csv"]:
        !wget -q -O {_f} {BASE_URL}/{_f}
    DATA_DIR = "."
else:
    DATA_DIR = os.path.join("..", "data") if os.path.isdir(os.path.join("..", "data")) else "."

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

RANDOM_STATE = 42

train = pd.read_csv(os.path.join(DATA_DIR, "02_train.csv"))
train = train.dropna(subset=["상품명"]).drop_duplicates().reset_index(drop=True)

X, y_text = train["상품명"].values, train["카테고리"].values
X_train, X_valid, y_train_text, y_valid_text = train_test_split(
    X, y_text, test_size=0.2, stratify=y_text, random_state=RANDOM_STATE
)

label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train_text)
y_valid = label_encoder.transform(y_valid_text)
N_CLASSES = len(label_encoder.classes_)


def train_model(seq_len=12, max_tokens=5000, embed_dim=64, split=None):
    """본문 4절에서 가장 좋았던 conv 구조로 학습하고 (모델, 벡터화 레이어, 검증 정확도, 학습 시간)을 돌려준다."""
    keras.utils.set_random_seed(RANDOM_STATE)
    kwargs = {} if split is None else {"split": split}

    vectorize = layers.TextVectorization(
        max_tokens=max_tokens, output_sequence_length=seq_len, **kwargs
    )
    vectorize.adapt(X_train)

    inputs = keras.Input(shape=(1,), dtype=tf.string)
    h = vectorize(inputs)
    h = layers.Embedding(max_tokens, embed_dim)(h)
    h = layers.Conv1D(128, 3, activation="relu")(h)
    h = layers.GlobalMaxPooling1D()(h)
    h = layers.Dropout(0.3)(h)
    h = layers.Dense(64, activation="relu")(h)
    outputs = layers.Dense(N_CLASSES, activation="softmax")(h)

    model = keras.Model(inputs, outputs)
    model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])

    started = time.time()
    model.fit(
        X_train, y_train, validation_data=(X_valid, y_valid),
        epochs=30, batch_size=64, verbose=0,
        callbacks=[keras.callbacks.EarlyStopping(monitor="val_loss", patience=3,
                                                 restore_best_weights=True)],
    )
    acc = model.evaluate(X_valid, y_valid, verbose=0)[1]
    return model, vectorize, acc, time.time() - started


print("준비 완료 · 학습", len(X_train), "· 검증", len(X_valid))

---

## 문제 1. `SEQ_LEN`을 4로 줄이면?

In [ ]:
for seq_len in [4, 12]:
    _, _, acc, _ = train_model(seq_len=seq_len)
    print(f"SEQ_LEN={seq_len:>2}  검증 정확도 {acc:.4f}")

print("\n앞 4단어만 남기면:")
for s in X_train[:5]:
    print(" ", " ".join(s.split()[:4]), "   <- 원본:", s)

**0.9077 → 0.8977. 약 1%p 떨어집니다. 생각보다 적게 떨어집니다.**

상품명의 구조를 보면 이유가 보입니다.

```
[무료배송] 한결식품 얼큰 컵라면 110g 2+1
  판매문구   브랜드  수식어  핵심어   용량  행사
```

**카테고리를 결정하는 핵심어가 대개 3~4번째 단어**라, 앞 4단어만 봐도 대부분 살아남습니다.
잘려나가는 것은 용량·행사 문구처럼 정보가 적은 뒷부분입니다.

그래도 1%p를 잃은 이유는 ① 판매 문구가 앞에 붙어 자리를 밀어낸 행과 ② 용량 단위에 남아 있던
약한 신호(01번 6절에서 확인)가 사라졌기 때문입니다.

**교훈:** 시퀀스 길이를 줄이면 **뒤가 잘립니다.** 중요한 정보가 뒤에 오는 데이터(예: 문장의 결론,
"~하지 않았다")라면 같은 조치가 훨씬 치명적입니다. **자르기 전에 어디에 정보가 있는지 확인하세요.**

---

## 문제 2. `MAX_TOKENS`를 실제 사전 크기에 맞추기

In [ ]:
for max_tokens in [1000, 5000]:
    model, _, acc, _ = train_model(max_tokens=max_tokens)
    print(f"MAX_TOKENS={max_tokens:>5}  정확도 {acc:.4f}  파라미터 {model.count_params():,}개")

**성능은 그대로인데 파라미터가 353,610 → 97,610개로 줄었습니다(약 1/3.6).**

실제 사전이 1,000개도 안 되는데 `Embedding(5000, 64)`을 만들면, **한 번도 쓰이지 않는 행이
4,000개** 생깁니다. 학습되지도 않고 파일 용량만 차지합니다.

`max_tokens`는 **사전 크기의 상한**일 뿐이라 넉넉히 잡아도 성능이 나빠지지는 않습니다.
다만 모델 파일이 커지고, 제출·재현이 느려집니다. **`adapt()` 후에 `len(get_vocabulary())`를
찍어보고 맞춰주는 것**이 좋은 습관입니다.

> 반대 방향의 실수도 있습니다. `max_tokens`를 실제 사전보다 **작게** 잡으면 드문 단어가
> `[UNK]`로 뭉개집니다. 그러면 학습 중에 `[UNK]`를 경험하게 되어 처음 보는 단어에 강해지는
> 부수 효과도 있습니다(일부러 그렇게 하기도 합니다).

---

## 문제 3. 임베딩 차원 16 / 64 / 256

In [ ]:
for dim in [16, 64, 256]:
    model, _, acc, elapsed = train_model(embed_dim=dim)
    print(f"embed_dim={dim:>3}  정확도 {acc:.4f}  파라미터 {model.count_params():,}개  {elapsed:.0f}초")

| 차원 | 정확도 | 파라미터 |
|---|---|---|
| 16 | 0.8957 | 95,178 |
| 64 | 0.9077 | 353,610 |
| 256 | 0.9067 | 1,387,338 |

**16은 부족하고, 256은 64보다 낫지 않습니다.** 파라미터는 4배가 됐는데 성능은 제자리입니다.

임베딩 차원은 **"단어 하나를 몇 개의 숫자로 표현할까"** 입니다. 너무 작으면 서로 다른 단어를
구분할 자리가 부족하고(과소적합), 너무 크면 학습할 값만 늘어 데이터가 부족해집니다(과적합).

**단어 1,000개짜리 사전에 256차원은 과합니다.** 실무의 대략적인 기준은 어휘가 수천~수만이면
50~300차원이고, 여기서는 어휘가 작으니 32~64면 충분합니다. **키운다고 좋아지지 않습니다.**

---

## 문제 4. 글자 단위로 자르기

In [ ]:
# 글자 단위는 시퀀스가 길어지므로 길이도 함께 늘립니다(상품명 최대 글자 수 참고).
print("상품명 글자 수 95% 지점:", int(pd.Series([len(s) for s in X_train]).quantile(0.95)))

_, _, acc_char, _ = train_model(seq_len=40, split="character")
_, _, acc_word, _ = train_model()

print(f"\n글자 단위 정확도 {acc_char:.4f}")
print(f"단어 단위 정확도 {acc_word:.4f}")

**글자 단위가 조금 더 좋습니다(0.9097 vs 0.9077).** 01번에서 문자 n-gram이 단어보다 나았던 것과
같은 결과입니다.

이유도 같습니다. **띄어쓰기와 표기 흔들림을 흡수**하기 때문입니다.
`치즈라면`과 `치즈 라면`은 단어 단위로는 완전히 다른 토큰이지만, 글자 단위로는 대부분의 글자를 공유합니다.
`Conv1D(128, 3)`이 **인접한 글자 3개 묶음**을 보므로, 사실상 학습되는 문자 3-gram입니다.

**대가는 시퀀스 길이입니다.** 단어 12개 대신 글자 40개를 처리하므로 계산량이 3배 이상 늘어납니다.
한국어에서 **글자 단위는 강력한 기본 선택지**이지만, 텍스트가 길어지면 비용이 빠르게 커집니다.

---

## 문제 5. 임베딩 공간에서 `라면`과 가까운 단어

In [ ]:
model, vectorize, acc, _ = train_model()

vocab = [str(w) for w in vectorize.get_vocabulary()]

# model.layers는 [입력, TextVectorization, Embedding, Conv1D, ...] 순서라 2번이 임베딩 층입니다.
# (model.summary()로 순서를 확인할 수 있습니다.) get_weights()[0]이 임베딩 행렬입니다.
# 행렬은 max_tokens행이지만 실제로 쓰이는 것은 사전 크기만큼이라, 나머지 행은 잘라냅니다.
embedding = model.layers[2].get_weights()[0][: len(vocab)]

# 코사인 유사도 = 정규화한 벡터의 내적
normalized = embedding / (np.linalg.norm(embedding, axis=1, keepdims=True) + 1e-9)


def 비슷한_단어(word, k=10):
    i = vocab.index(word)
    sim = normalized @ normalized[i]
    return [(vocab[j], round(float(sim[j]), 2)) for j in np.argsort(-sim) if j != i][:k]


for w in ["라면", "요거트"]:
    print(f"[{w}]과 가까운 단어")
    for word, score in 비슷한_단어(w):
        print(f"   {word:<10} {score}")
    print()

**의미가 비슷한 단어가 실제로 가깝습니다.**

```
[라면]과 가까운 단어
   칼국수 0.88 · 라볶이 0.88 · 컵라면 0.81 · 우동 0.78 · 짜장면 0.73 ...
```

`칼국수`, `라볶이`, `우동`은 **모두 `라면류` 카테고리의 핵심어**입니다. 모델은 사전에서 이 단어들의
뜻을 배운 적이 없습니다. **"같은 카테고리를 예측하게 만드는 단어끼리 가까워지도록" 역전파가
벡터를 밀어놓은 것**입니다. 이것이 [임베딩](https://github.com/karzit/temp/blob/master/glossary.md#embedding)이 하는 일입니다.

**단, 여기서의 "의미"는 이 문제에 한정됩니다.** 이 임베딩은 "카테고리 분류에 유용한가"라는 기준으로만
학습되었으므로, `5컵` 같은 단어도 라면과 가까워집니다(라면에만 붙는 단위라서요).
범용적인 한국어 의미 공간을 원한다면 사전 학습된 임베딩(Word2Vec, FastText, KoBERT)을 씁니다.
[RAG 실습](https://github.com/karzit/temp/blob/master/notebooks/rag-pipeline-practice/README.md)에서 쓴 문장 임베딩이 그런 경우입니다.

---

## 문제 6. TF-IDF 모델과 앙상블

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.pipeline import make_pipeline

tfidf_model = make_pipeline(
    TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 3)),
    LogisticRegression(max_iter=1000, random_state=RANDOM_STATE),
).fit(X_train, y_train_text)

# 두 모델의 클래스 순서를 맞춰야 확률을 더할 수 있습니다.
proba_tfidf = np.zeros((len(X_valid), N_CLASSES))
for k, cls in enumerate(tfidf_model.classes_):
    proba_tfidf[:, list(label_encoder.classes_).index(cls)] = tfidf_model.predict_proba(X_valid)[:, k]

proba_nn = model.predict(X_valid, verbose=0)
proba_mix = (proba_tfidf + proba_nn) / 2

for name, proba in [("TF-IDF", proba_tfidf), ("신경망", proba_nn), ("앙상블", proba_mix)]:
    print(f"{name:<8} 정확도 {accuracy_score(y_valid, proba.argmax(axis=1)):.4f}")

틀림_tfidf = proba_tfidf.argmax(axis=1) != y_valid
틀림_nn = proba_nn.argmax(axis=1) != y_valid
print(f"\n둘 다 틀림 {int((틀림_tfidf & 틀림_nn).sum())}건"
      f" · TF-IDF만 틀림 {int((틀림_tfidf & ~틀림_nn).sum())}건"
      f" · 신경망만 틀림 {int((틀림_nn & ~틀림_tfidf).sum())}건")

**앙상블이 두 모델보다 낫습니다(0.9067 / 0.9077 → 0.9127).**

숫자를 보면 왜 이득인지 분명합니다.

| | 건수 |
|---|---|
| 둘 다 틀림 | 67 |
| TF-IDF만 틀림 | 26 |
| 신경망만 틀림 | 25 |

**두 모델이 서로 다른 곳에서 틀립니다.** 한쪽만 틀린 51건은 다른 쪽이 확신을 갖고 맞히면
평균에서 살아납니다. 반대로 **둘 다 틀린 67건은 앙상블로도 구제되지 않습니다** — 핵심어가 없거나
라벨이 틀린 행들이라, 어떤 모델을 더해도 못 맞힙니다.

**앙상블이 이득인 조건은 "모델들이 서로 다른 실수를 하는 것"입니다.** 같은 데이터를
같은 방식으로 본 모델 두 개를 더해봐야 소용이 없습니다. 여기서는 한쪽은 문자 조각의 통계를,
다른 쪽은 학습된 임베딩을 보므로 실수의 방향이 달랐습니다.

> **시험에서 쓸 수 있나요?** 제출하는 모델 파일은 하나지만, **예측 csv는 앙상블 결과로 만들 수
> 있습니다.** 다만 채점에서 "제출한 코드로 실행했을 때 재현되는지"를 보므로, 노트북에 두 모델의
> 학습과 앙상블 과정이 **모두 들어 있어야** 합니다. 그리고 두 모델을 학습시키는 만큼 시간이 듭니다.
> **먼저 단일 모델로 목표 성능을 확보한 뒤** 시간이 남으면 시도하세요.

---

## 정리

| 문제 | 배운 것 |
|---|---|
| 1 | 시퀀스 길이를 줄이면 **뒤가 잘린다**. 정보가 어디에 있는지 보고 정한다 |
| 2 | `max_tokens`는 상한일 뿐. **실제 사전 크기에 맞추면 모델이 1/3로 줄어든다** |
| 3 | 임베딩 차원은 **키운다고 좋아지지 않는다**. 어휘 크기에 맞춘다 |
| 4 | 한국어에서 **글자 단위는 강력한 기본 선택지**. 대신 시퀀스가 길어진다 |
| 5 | 임베딩은 **분류에 유용한 방향으로** 단어를 배치한다. 범용 의미 공간이 아니다 |
| 6 | 앙상블의 조건은 **서로 다른 실수**. 둘 다 틀리는 행은 구제되지 않는다 |